# Temporal Multiplex Directed Networks for the Semiconductor Industry

A Temporal Multiplex Directed Network $\mathcal{M}$ is defined as a sequence of layers $L = \{L_1, L_2, \dots, L_M\}$, where each layer represents a different type of interaction (Financial, Supply Chain, etc.) over time steps $t \in \{1, \dots, T\}$.

The state of the network at any time $t$ is represented by a Supra-Adjacency Tensor $\mathcal{A}$: $$\mathcal{A}_{i,j, \alpha}(t)$$
Where: 
$i, j \in \{1, \dots, N\}$ are the semiconductor companies (nodes). 
$\alpha \in \{1, \dots, M\}$ is the specific layer (e.g., $\alpha=1$ for the Financial Layer, $\alpha=2$ for the Supply Chain Layer, $\alpha=3$ for the Ownership Layer). $t$ is the temporal window (e.g., the specific week).

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import matplotlib.pyplot as plt
import networkx as nx
from py_scripts.project2 import financial_layer as fl

## Financial Layer

TODO WRITE A SUMMARY OF THE STEPS DONE AND WHAT WAS ACHIEVED


### DATA ACQUISITION

In [ ]:
foundries = {
    "TSM": "Taiwan Semiconductor Manufacturing Company Limited",
    "SSNLF": "Samsung Electronics Co., Ltd.",
    "INTC": "Intel Corporation",
    "UMC": "United Microelectronics Corporation",
    "GFS": "GlobalFoundries Inc.",
}

fabless_designers = {
    "NVDA": "NVIDIA Corporation",
    "AMD": "Advanced Micro Devices, Inc.",
    "AVGO": "Broadcom Inc.",
    "ARM": "Arm Holdings plc",
    "QCOM": "QUALCOMM Incorporated",
    "MRVL": "Marvell Technology, Inc.",
    "ALAB": "Astera Labs, Inc.",
}

memory = {
    "MU": "Micron Technology, Inc.",
}

wfe = {
    "ASML": "ASML Holding N.V.",
    "AMAT": "Applied Materials, Inc.",
    "LRCX": "Lam Research Corporation",
    "KLAC": "KLA Corporation",
    "TOELY": "Tokyo Electron Limited",
    "ADVNF": "Advantest Corporation",
    "TER": "Teradyne, Inc.",
    "SNPS": "Synopsys, Inc.",
    "CDNS": "Cadence Design Systems, Inc.",
}

osat_packaging = {
    "ASX": "ASE Technology Holding Co., Ltd.",
    "AMKR": "Amkor Technology, Inc.",
}

analog_auto_power = {
    "TXN": "Texas Instruments Incorporated",
    "ADI": "Analog Devices, Inc.",
    "NXPI": "NXP Semiconductors N.V.",
    "STNE": "STMicroelectronics N.V.",
    "ON": "ON Semiconductor Corporation",
    "IFNNY": "Infineon Technologies AG",
    "MCHP": "Microchip Technology Incorporated"
}

# Organize spheres
spheres = {
    "Foundries": foundries,
    "Fabless Designers": fabless_designers,
    "Memory": memory,
    "WFE (Equipment)": wfe,
    "OSAT & Packaging": osat_packaging,
    "Analog/Auto/Power": analog_auto_power,
}

In [62]:
# Download and visualize each sphere with 1h interval
for sphere_name, tickers_dict in spheres.items():
    data = yf.download(list(tickers_dict.keys()), start="2025-01-01", interval='1h', prepost=False, progress=False)['Close']
    
    fig = px.line(data.reset_index(), x='Datetime', y=data.columns,
                  title=f'{sphere_name} - Close Price (Since 2025, 1h Intervals)',
                  labels={'value': 'Close Price (USD)', 'variable': 'Ticker'})
    fig.update_layout(xaxis_title='Datetime', yaxis_title='Price (USD)')
    fig.show()

In [66]:
# Compute and visualize returns for each sphere
total_returns = pd.DataFrame()
for sphere_name, tickers_dict in spheres.items():
    data = yf.download(list(tickers_dict.keys()), start="2025-01-01", interval='1h', prepost=False, progress=False)['Close']
    data = data.replace(0, np.nan).ffill()
    returns = np.log(data / data.shift(1)).dropna()

    total_returns = pd.concat([total_returns, returns], axis=1)

    fig = px.line(returns.reset_index(), x='Datetime', y=returns.columns,
                  title=f'{sphere_name} - Returns (Since 2025, 1h Intervals)',
                  labels={'value': 'Returns', 'variable': 'Ticker'})
    fig.update_layout(xaxis_title='Datetime', yaxis_title='Returns')
    fig.show()

C:\Users\Youdas Yessad\AppData\Local\Temp\ipykernel_33040\928971512.py:8: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  total_returns = pd.concat([total_returns, returns], axis=1)


C:\Users\Youdas Yessad\AppData\Local\Temp\ipykernel_33040\928971512.py:8: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  total_returns = pd.concat([total_returns, returns], axis=1)


In [67]:
# Diagnostic for WFE prices & returns
wfe_tickers = list(wfe.keys())
print('WFE tickers:', wfe_tickers)

raw = yf.download(wfe_tickers, start="2025-01-01", interval='1h', progress=False)
print('download type:', type(raw))
print('columns:', getattr(raw, 'columns', None))

# Extract Close prices robustly
if isinstance(raw, pd.DataFrame) and isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close'] if 'Close' in raw.columns.levels[0] else raw
elif isinstance(raw, pd.Series):
    prices = raw.to_frame('Close')
else:
    prices = pd.DataFrame(raw)

print('prices type, shape:', type(prices), getattr(prices, 'shape', None))
print('columns before dropna:', list(prices.columns))

# Identify completely-empty tickers
all_nan_cols = prices.columns[prices.isna().all()].tolist()
print('all-NaN columns (no data):', all_nan_cols)

prices = prices.dropna(axis=1, how='all').replace(0, np.nan)
prices_filled = prices.ffill().bfill()

returns = np.log(prices_filled / prices_filled.shift(1))

inf_counts = np.isinf(returns).sum()
nan_counts = returns.isna().sum()
extreme_cols = returns.columns[(returns.abs() > 1).any()].tolist()

print('prices head:')
print(prices.head())
print('returns shape:', returns.shape)
print('columns with inf counts >0:\n', inf_counts[inf_counts>0])
print('columns with NaN counts >0:\n', nan_counts[nan_counts>0])
print('columns with abs(return)>1 at least once:', extreme_cols)

WFE tickers: ['ASML', 'AMAT', 'LRCX', 'KLAC', 'TOELY', 'ADVNF', 'TER', 'SNPS', 'CDNS']
download type: <class 'pandas.DataFrame'>
columns: MultiIndex([( 'Close', 'ADVNF'),
            ( 'Close',  'AMAT'),
            ( 'Close',  'ASML'),
            ( 'Close',  'CDNS'),
            ( 'Close',  'KLAC'),
            ( 'Close',  'LRCX'),
            ( 'Close',  'SNPS'),
            ( 'Close',   'TER'),
            ( 'Close', 'TOELY'),
            (  'High', 'ADVNF'),
            (  'High',  'AMAT'),
            (  'High',  'ASML'),
            (  'High',  'CDNS'),
            (  'High',  'KLAC'),
            (  'High',  'LRCX'),
            (  'High',  'SNPS'),
            (  'High',   'TER'),
            (  'High', 'TOELY'),
            (   'Low', 'ADVNF'),
            (   'Low',  'AMAT'),
            (   'Low',  'ASML'),
            (   'Low',  'CDNS'),
            (   'Low',  'KLAC'),
            (   'Low',  'LRCX'),
            (   'Low',  'SNPS'),
            (   'Low',   'TER'),
    

### PCA on Returns

The denoising of semiconductor returns relies on the spectral decomposition of the empirical correlation matrix $C$, where the returns are first standardized to unit variance. $$C = \frac{1}{T} Z^T Z = V \Lambda V^T$$ By applying the Marchenko-Pastur theorem, we identify a theoretical noise boundary $\lambda_{max}$ that separates structural market signals from random eigenvalues. $$\lambda_{max} = \sigma^2 (1 + \sqrt{N/T})^2$$  We perform a low-rank reconstruction by projecting the returns $Z$ onto only the $k$ most significant eigenvectors, effectively filtering the data. $$\hat{Z} = (ZV_{sig})V_{sig}^T$$ 
This filtering prevents Sparse VAR Lasso from overfitting to random artifacts, ensuring that the lead-lag edges fed into the GNN represent true structural dependencies rather than coincidental noise.


In [64]:
window = 35 * 7 
assets_num = 38
all_denoised_windows = []

for i in range(window, len(total_returns), 7):
    
    window_returns = total_returns.iloc[i-window:i]
    
    # Standardize the window returns
    window_mean = window_returns.mean()
    window_std = window_returns.std()
    standardized_slice = (window_returns - window_mean) / window_std
    
    eigenvalues, eigenvectors = fl.PCA(standardized_slice)

    q = window / assets_num
    lambda_max = (1 + np.sqrt(1/q))**2
    print(f"Window {i//7}: λ_max = {lambda_max:.4f}")
    significant_components = np.sum(eigenvalues > lambda_max)
    
    # Project the returns on the eigenvectors
    pca_projections = standardized_slice.values @ eigenvectors
    pca_projections[:, significant_components:] = 0

    denoised_standardized = pca_projections @ eigenvectors.T
    
    # Unstandardize the denoised data
    denoised_final = (denoised_standardized * window_std.values) + window_mean.values
    
    df_denoised = pd.DataFrame(denoised_final, 
                               index=window_returns.index, 
                               columns=window_returns.columns)
    all_denoised_windows.append(df_denoised)

LinAlgError: Array must not contain infs or NaNs

### Breaking Symmetry in the Financial Layer

To transform a standard undirected correlation into a Directed Lead-Lag Network, we define the directed adjacency matrix $A^{(dir)}$ using a time-shifted correlation.

For any two assets $i$ and $j$, the directed edge weight $E_{i \to j}$ is calculated as:$$E_{i \to j}(t) = \text{corr}(R_{i, t}, R_{j, t+1})$$
Conversely, the influence of $j$ on $i$ is:$$E_{j \to i}(t) = \text{corr}(R_{j, t}, R_{i, t+1})$$
In this construction, $A^{(dir)}$ is asymmetric ($E_{i \to j} \neq E_{j \to i}$), representing the directional flow of information from a "leader" to a "lagger."